## 라이브러리 및 상수 선언

In [1]:
# pip install google_play_scraper

In [2]:
from google_play_scraper import reviews, Sort
import time
import pandas as pd

In [3]:
PACKAGE_LIST = ['com.sampleapp','com.coupang.mobile.eats','com.fineapp.yogiyo','com.shinhan.o2o'] # 배민, 쿠팡이츠, 요기요, 땡겨요
PACKAGE_NAME_KOR = ['배달의민족', '쿠팡이츠', '요기요', '땡겨요']
PACKAGE_NAME_ENG = ['baemin', 'coupangeats', 'yogiyo', 'ddangyo']
PACKAGE_NUM = 3
NUM_DATA = 7000

In [4]:
APP_ID = PACKAGE_LIST[PACKAGE_NUM]
APP_NAME_KOR = PACKAGE_NAME_KOR[PACKAGE_NUM]
APP_NAME_ENG = PACKAGE_NAME_ENG[PACKAGE_NUM]

## 비공식 API를 활용한 데이터 수집

In [5]:
all_reviews = []
token = None
seen = set()

In [6]:
while True: # 한번의 호출가능한 수가 한정되어있으므로 반복해야함. count=10,000이라고해서 10,000개 가져오는거 아님.
    batch, token = reviews(
        APP_ID,
        lang="ko", # 언어
        country="kr", # 국가
        count=200,
        sort=Sort.NEWEST,
        continuation_token=token
    )
    # 중복 제거 (수집 중 리뷰가 추가되어 중복된 리뷰가 들어갈 수 있음.)
    add = 0 # 추가 여부 확인
    for r in batch:
        rid = r.get("reviewId")
        if rid not in seen:
            seen.add(rid)
            all_reviews.append(r)
            add += 1
    if add == 0:
        break
    # print(len(all_reviews))
    # print(type(review[0]))
    # print(type(review[0][0]))
    # print(len(review[0]))
    # print(review[0][0].keys())
    # ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

    if len(all_reviews) >= NUM_DATA:
        all_reviews = all_reviews[:NUM_DATA]
        break

    time.sleep(0.5) # 너무 빠르게 호출하면 에러 발생할 수 있음.

In [7]:
print("수집한 리뷰 수:", len(all_reviews))

수집한 리뷰 수: 7000


## 간단한 EDA

In [8]:
df = pd.DataFrame(all_reviews)

In [9]:
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,89953a30-99f8-4cd6-b933-3f192c22c5bb,월천일백,https://play-lh.googleusercontent.com/a/ACg8oc...,배민의 대항마,5,0,1.9.3,2025-12-15 08:40:46,None,NaT,1.9.3
1,51f5cd17-73a5-49e5-aad5-a1421cb753c4,이규철,https://play-lh.googleusercontent.com/a/ACg8oc...,호두빵 실제로 보면서 든 생각 와 어 어으 돈의가치가 이리도 떨어졌단말인가 ㅜㅜ,5,0,None,2025-12-15 00:04:14,None,NaT,None
2,6add68da-020f-4fe2-a86e-b006568e790e,유제노비아,https://play-lh.googleusercontent.com/a/ACg8oc...,주말 늦은 저녁 잘 먹었네요~,5,0,1.9.3,2025-12-14 23:56:13,None,NaT,1.9.3
3,eea9e5fa-fc87-400c-adc5-16bacb247f85,김희정,https://play-lh.googleusercontent.com/a/ACg8oc...,지역화폐 사용 좋아요. 쿠폰 좋아요,5,0,1.9.3,2025-12-14 23:07:41,None,NaT,1.9.3
4,86641930-3790-4c24-bf48-8b0a1f0b07ec,Donsoon Choi,https://play-lh.googleusercontent.com/a-/ALV-U...,배달현황이 지도로 표시되면 좋겠네요.,5,0,1.9.3,2025-12-14 23:04:35,None,NaT,1.9.3


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              7000 non-null   object        
 1   userName              7000 non-null   object        
 2   userImage             7000 non-null   object        
 3   content               7000 non-null   object        
 4   score                 7000 non-null   int64         
 5   thumbsUpCount         7000 non-null   int64         
 6   reviewCreatedVersion  6320 non-null   object        
 7   at                    7000 non-null   datetime64[ns]
 8   replyContent          1898 non-null   object        
 9   repliedAt             1898 non-null   datetime64[ns]
 10  appVersion            6320 non-null   object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 601.7+ KB


In [11]:
len(df['reviewId'].unique())

7000

In [12]:
# df['reviewCreatedVersion'].unique()

In [13]:
# df['appVersion'].unique()

In [14]:
(df['reviewCreatedVersion'].fillna('MISSING')  == df['appVersion'].fillna('MISSING')).sum() # nan == nan 은 false이므로 missing으로 결측치 처리 후 비교

np.int64(7000)

In [15]:
df['app'] = APP_NAME_KOR
df['platform'] = 'playstore'
# df['sentiment'] = 1 if (df['score'] > 3) else 0
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app,platform
0,89953a30-99f8-4cd6-b933-3f192c22c5bb,월천일백,https://play-lh.googleusercontent.com/a/ACg8oc...,배민의 대항마,5,0,1.9.3,2025-12-15 08:40:46,None,NaT,1.9.3,땡겨요,playstore
1,51f5cd17-73a5-49e5-aad5-a1421cb753c4,이규철,https://play-lh.googleusercontent.com/a/ACg8oc...,호두빵 실제로 보면서 든 생각 와 어 어으 돈의가치가 이리도 떨어졌단말인가 ㅜㅜ,5,0,None,2025-12-15 00:04:14,None,NaT,None,땡겨요,playstore
2,6add68da-020f-4fe2-a86e-b006568e790e,유제노비아,https://play-lh.googleusercontent.com/a/ACg8oc...,주말 늦은 저녁 잘 먹었네요~,5,0,1.9.3,2025-12-14 23:56:13,None,NaT,1.9.3,땡겨요,playstore
3,eea9e5fa-fc87-400c-adc5-16bacb247f85,김희정,https://play-lh.googleusercontent.com/a/ACg8oc...,지역화폐 사용 좋아요. 쿠폰 좋아요,5,0,1.9.3,2025-12-14 23:07:41,None,NaT,1.9.3,땡겨요,playstore
4,86641930-3790-4c24-bf48-8b0a1f0b07ec,Donsoon Choi,https://play-lh.googleusercontent.com/a-/ALV-U...,배달현황이 지도로 표시되면 좋겠네요.,5,0,1.9.3,2025-12-14 23:04:35,None,NaT,1.9.3,땡겨요,playstore


app : 배달앱 이름

platform : 스토어 종류(플레이스토어/앱스토어)

reviewId : id(중복비교를 위해 사용)

userName	: 리뷰작성한 사용자 이름

userImage : 리뷰작성한 사용자 프로필 이미지

content : 리뷰

score : 별점

thumbsUpCount : 좋아요수

reviewCreatedVersion : 리뷰가 작성될 당시 사용자가 쓰고 있던 앱 버전

at : 리뷰 작성 날짜

replyContent : 리뷰에 대해 앱 운영사(개발사)가 남긴 공식 답변 내용

repliedAt : 답글을 남긴 날짜

appVersion : 리뷰 수집 당시 구글 플레이에 노출되는 현재 앱 버전 정보

reviewCreatedVersion랑 appVersion은 다를 수 있지만 수집된 데이터에 한해서는 완전히 같은 정보를 가지고 있음.

저장할 컬럼 [app, platform, reviewId, userName, content, score, thumbsUpCount, at]

## CSV 파일 저장

In [16]:
columns = ['app', 'platform', 'reviewId', 'userName', 'content', 'score', 'thumbsUpCount', 'at']#, 'sentiment']

In [17]:
df[columns].to_csv(f'{APP_NAME_ENG}_reviews_playstore_{NUM_DATA}.csv', index=False, encoding="utf-8-sig") # utf-8-sig:윈도우+엑셀에서 한글 깨짐 방지